# Modelo jerárquico para `receptions` y `receiving_yards` -- ¿se repite la mejora?

## Qué responde este notebook

`24_modelo_jerarquico_binomial_negativa.ipynb` mostró que un modelo jerárquico (efecto aleatorio
por jugador) con Binomial Negativa da el mejor R² de todos los intentos para `receiving_tds`. La
pregunta directa: ¿el mismo enfoque mejora también `receptions` y `receiving_yards`?

**Resultado honesto, no el esperado**: no. En ambos targets el modelo jerárquico queda por debajo
del XGBoost ya guardado. Se documenta el porqué -- la mejora en `receiving_tds` no vino de
"jerárquico es mejor en general", vino de corregir una discrepancia real de distribución
(`reg:tweedie` mal especificado para un conteo con 82% de ceros). En los otros 2 targets esa
discrepancia no existe de la misma forma, y la flexibilidad no paramétrica de un árbol le gana a
la estructura más rígida de un modelo lineal jerárquico.

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features, modeling

import numpy as np
import pandas as pd
import bambi as bmb
import arviz as az
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
import time

pd.set_option("display.width", 160)

wr = features.construir_tabla_modelado(range(2016, 2026))
features_ridge = features.columnas_por_modelo("ridge")
train_mask = wr["season"].between(2016, 2021)
test_mask = wr["season"].between(2024, 2025)

cols_numericas = [c for c in features_ridge if c != "anios_experiencia_bucket"]

def imputar(df, ajustar, imp_num, imp_cat):
    fn_num = imp_num.fit_transform if ajustar else imp_num.transform
    fn_cat = imp_cat.fit_transform if ajustar else imp_cat.transform
    num = pd.DataFrame(fn_num(df[cols_numericas]), index=df.index, columns=cols_numericas)
    cat = pd.DataFrame(fn_cat(df[["anios_experiencia_bucket"]]), index=df.index, columns=["anios_experiencia_bucket"])
    return pd.concat([num, cat], axis=1)[features_ridge]


## 1. `receptions` -- Binomial Negativa (mismo criterio que `receiving_tds`: conteo entero,
sobredispersión moderada var/media=2.35, 21% de semanas en cero)

In [2]:
target = "receptions"
imp_num_rec, imp_cat_rec = SimpleImputer(strategy="median"), SimpleImputer(strategy="most_frequent")
X_train_rec = imputar(wr.loc[train_mask, features_ridge], True, imp_num_rec, imp_cat_rec)
datos_train_rec = X_train_rec.copy()
datos_train_rec[target] = wr.loc[train_mask, target].to_numpy()
datos_train_rec["player_id"] = wr.loc[train_mask, "player_id"].astype(str).to_numpy()

formula_rec = f"{target} ~ " + " + ".join(features_ridge) + " + (1|player_id)"
modelo_rec = bmb.Model(formula_rec, datos_train_rec, family="negativebinomial")

t0 = time.time()
idata_rec = modelo_rec.fit(draws=500, tune=500, chains=4, cores=4, random_seed=0, progressbar=False)
print(f"tiempo de ajuste: {time.time()-t0:.0f}s")

resumen_rec = az.summary(idata_rec)
print(f"r_hat max: {resumen_rec['r_hat'].max():.4f} | parametros > 1.01: {(resumen_rec['r_hat']>1.01).sum()} de {len(resumen_rec)}")


Initializing NUTS using jitter+adapt_diag...


/usr/local/lib/python3.13/site-packages/pytensor/tensor/rewriting/elemwise.py:1004: UserWarning: Loop fusion failed because the resulting node would exceed the kernel argument limit.
  warn(


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [alpha, Intercept, anios_experiencia, catch_rate_career_avg, depth_team, draft_pick, edad, fantasy_points_ppr_last3_avg, fantasy_points_ppr_last5_avg, fantasy_points_ppr_season_avg, receiving_10_last3_avg, receiving_16_last5_avg, receiving_20_last3_avg, receiving_air_yards_season_avg, receiving_epa_last3_avg, receiving_first_downs_career_avg, receiving_first_downs_last5_avg, receiving_tds_season_avg, receiving_yards_career_avg, receiving_yards_last3_avg, receiving_yards_season_avg, receptions_career_avg, receptions_last3_avg, receptions_last5_avg, receptions_season_avg, targets_last3_avg, targets_last5_avg, targets_season_avg, wopr_last3_avg, wopr_last5_avg, anios_experiencia_sq, anios_experiencia_bucket, 1|player_id_sigma, 1|player_id_offset]


Sampling 4 chains for 500 tune and 500 draw iterations (2_000 + 2_000 draws total) took 381 seconds.


The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


tiempo de ajuste: 396s


r_hat max: 1.0430 | parametros > 1.01: 59 de 540


In [3]:
X_test_rec = imputar(wr.loc[test_mask, features_ridge], False, imp_num_rec, imp_cat_rec)
datos_test_rec = X_test_rec.copy()
datos_test_rec["player_id"] = wr.loc[test_mask, "player_id"].astype(str).to_numpy()

pred_rec = modelo_rec.predict(idata_rec, data=datos_test_rec, kind="response_params", inplace=False, sample_new_groups=True)
pred_media_rec = pred_rec.posterior["mu"].mean(dim=["chain","draw"]).to_numpy()
y_test_rec = wr.loc[test_mask, target].to_numpy()
m_rec_jerarquico = modeling.metricas(pd.Series(y_test_rec), pd.Series(pred_media_rec))

# mismos hiperparametros ya guardados del XGBoost puntual, para comparar limpio
modelo_xgb_rec = XGBRegressor(max_depth=4, n_estimators=200, learning_rate=0.03, random_state=0, n_jobs=-1)
modelo_xgb_rec.fit(wr.loc[train_mask, features.columnas_por_modelo("arbol")], wr.loc[train_mask, target])
pred_xgb_rec = np.clip(modelo_xgb_rec.predict(wr.loc[test_mask, features.columnas_por_modelo("arbol")]), 0, None)
m_rec_xgb = modeling.metricas(pd.Series(y_test_rec), pd.Series(pred_xgb_rec))

print(pd.DataFrame({"XGBoost (ya guardado)": m_rec_xgb, "Jerarquico + Binomial Negativa": m_rec_jerarquico}).T.round(4).to_string())


/tmp/ipykernel_7/1194447468.py:5: FutureWarning: 'sample_new_groups' is deprecated and will be removed in a future version. New groups will be handled automatically.
  pred_rec = modelo_rec.predict(idata_rec, data=datos_test_rec, kind="response_params", inplace=False, sample_new_groups=True)


                                   mae    rmse      r2
XGBoost (ya guardado)           1.3647  1.8257  0.4532
Jerarquico + Binomial Negativa  1.4777  1.9748  0.3602


**Peor en las 2 métricas**, y con convergencia claramente más débil que `receiving_tds`
(más parámetros sobre el umbral de r_hat). A diferencia de touchdowns, `receptions` no tenía una
discrepancia de distribución tan marcada en el modelo anterior (error cuadrático sobre un conteo
de 0-15 con 21% de ceros es una aproximación razonable, no como Tweedie sobre un conteo de 82%
ceros) -- la ganancia de usar la distribución "correcta" es menor aquí, y no alcanza a compensar
la pérdida de flexibilidad no paramétrica de un árbol.

## 2. `receiving_yards` -- un bug real encontrado en el camino

`receiving_yards` es continuo, no un conteo -- la Binomial Negativa no aplica. La opción natural
era una transformación logarítmica + familia Gaussiana (`log1p`, igual que otros proyectos de
datos con cola larga a la derecha). Antes de usarla, vale revisar si el supuesto de "siempre
positivo" es cierto.

In [4]:
negativos = wr.loc[train_mask & (wr["receiving_yards"] < 0)]
print(f"filas de train con receiving_yards < 0: {len(negativos)} de {train_mask.sum()}")
print(negativos[["player_display_name", "season", "week", "receptions", "receiving_yards"]].head(5).to_string())


filas de train con receiving_yards < 0: 67 de 13795
     player_display_name  season  week  receptions  receiving_yards
371          Eddie Royal    2016    12           1               -6
1199    Demaryius Thomas    2019     2           1               -1
1591        Randall Cobb    2019    15           1               -3
1899     Dontrelle Inman    2020     5           1               -2
2223     Jermaine Kearse    2018     2           1               -1


Real, no un error de datos: son jugadas donde el receptor fue tackleado detrás de la línea
de golpeo (ej. un *screen pass* o motion en jet sweep que termina en pérdida) -- un jugador puede
terminar el partido con yardas de recepción netas negativas. `np.log1p()` no soporta valores
menores a -1 (de las 61 filas negativas, varias llegan hasta -9) -- se usa una transformación
logarítmica con signo (`sign(x) * log1p(|x|)`), que sí maneja negativos y mapea 0 → 0 exactamente.

In [5]:
target = "receiving_yards"
imp_num_yds, imp_cat_yds = SimpleImputer(strategy="median"), SimpleImputer(strategy="most_frequent")
X_train_yds = imputar(wr.loc[train_mask, features_ridge], True, imp_num_yds, imp_cat_yds)
datos_train_yds = X_train_yds.copy()
y_train_crudo = wr.loc[train_mask, target].to_numpy()
datos_train_yds[f"{target}_log"] = np.sign(y_train_crudo) * np.log1p(np.abs(y_train_crudo))
datos_train_yds["player_id"] = wr.loc[train_mask, "player_id"].astype(str).to_numpy()

formula_yds = f"{target}_log ~ " + " + ".join(features_ridge) + " + (1|player_id)"
modelo_yds = bmb.Model(formula_yds, datos_train_yds, family="gaussian")

t0 = time.time()
idata_yds = modelo_yds.fit(draws=500, tune=500, chains=4, cores=4, random_seed=0, progressbar=False)
print(f"tiempo de ajuste: {time.time()-t0:.0f}s")

resumen_yds = az.summary(idata_yds)
print(f"r_hat max: {resumen_yds['r_hat'].max():.4f} | parametros > 1.01: {(resumen_yds['r_hat']>1.01).sum()} de {len(resumen_yds)}")


Initializing NUTS using jitter+adapt_diag...


/usr/local/lib/python3.13/site-packages/pytensor/tensor/rewriting/elemwise.py:1004: UserWarning: Loop fusion failed because the resulting node would exceed the kernel argument limit.
  warn(


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [sigma, Intercept, anios_experiencia, catch_rate_career_avg, depth_team, draft_pick, edad, fantasy_points_ppr_last3_avg, fantasy_points_ppr_last5_avg, fantasy_points_ppr_season_avg, receiving_10_last3_avg, receiving_16_last5_avg, receiving_20_last3_avg, receiving_air_yards_season_avg, receiving_epa_last3_avg, receiving_first_downs_career_avg, receiving_first_downs_last5_avg, receiving_tds_season_avg, receiving_yards_career_avg, receiving_yards_last3_avg, receiving_yards_season_avg, receptions_career_avg, receptions_last3_avg, receptions_last5_avg, receptions_season_avg, targets_last3_avg, targets_last5_avg, targets_season_avg, wopr_last3_avg, wopr_last5_avg, anios_experiencia_sq, anios_experiencia_bucket, 1|player_id_sigma, 1|player_id_offset]


Sampling 4 chains for 500 tune and 500 draw iterations (2_000 + 2_000 draws total) took 43 seconds.


The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


tiempo de ajuste: 52s


r_hat max: 1.0156 | parametros > 1.01: 7 de 540


In [6]:
X_test_yds = imputar(wr.loc[test_mask, features_ridge], False, imp_num_yds, imp_cat_yds)
datos_test_yds = X_test_yds.copy()
datos_test_yds["player_id"] = wr.loc[test_mask, "player_id"].astype(str).to_numpy()

pred_yds = modelo_yds.predict(idata_yds, data=datos_test_yds, kind="response_params", inplace=False, sample_new_groups=True)
pred_log_yds = pred_yds.posterior["mu"].mean(dim=["chain","draw"]).to_numpy()
pred_media_yds = np.clip(np.sign(pred_log_yds) * np.expm1(np.abs(pred_log_yds)), 0, None)
y_test_yds = wr.loc[test_mask, target].to_numpy()
m_yds_jerarquico = modeling.metricas(pd.Series(y_test_yds), pd.Series(pred_media_yds))

modelo_xgb_yds = XGBRegressor(max_depth=3, n_estimators=200, learning_rate=0.03, random_state=0, n_jobs=-1)
modelo_xgb_yds.fit(wr.loc[train_mask, features.columnas_por_modelo("arbol")], wr.loc[train_mask, target])
pred_xgb_yds = np.clip(modelo_xgb_yds.predict(wr.loc[test_mask, features.columnas_por_modelo("arbol")]), 0, None)
m_yds_xgb = modeling.metricas(pd.Series(y_test_yds), pd.Series(pred_xgb_yds))

print(pd.DataFrame({"XGBoost (ya guardado)": m_yds_xgb, "Jerarquico Gaussiano (log con signo)": m_yds_jerarquico}).T.round(4).to_string())


/tmp/ipykernel_7/2885173176.py:5: FutureWarning: 'sample_new_groups' is deprecated and will be removed in a future version. New groups will be handled automatically.
  pred_yds = modelo_yds.predict(idata_yds, data=datos_test_yds, kind="response_params", inplace=False, sample_new_groups=True)


                                          mae     rmse      r2
XGBoost (ya guardado)                 20.6502  27.9639  0.3664
Jerarquico Gaussiano (log con signo)  21.3410  32.4998  0.1441


**Notablemente peor en R²** (0.14 vs 0.37) aunque la convergencia es buena (similar a
`receiving_tds`, no es un problema de ajuste). La razón más probable: `receiving_yards` depende de
interacciones no lineales entre features (volumen × eficiencia × contexto de equipo) que un
modelo lineal aditivo -- incluso en escala logarítmica -- no captura tan bien como un ensamble de
árboles. La transformación logarítmica ayuda con el sesgo de la distribución, pero no resuelve la
falta de flexibilidad estructural del modelo.

## Conclusión

| Target | XGBoost (MAE / R²) | Jerárquico (MAE / R²) | ¿Mejora? |
|---|---|---|---|
| `receiving_tds` | 0.249 / -0.041 | 0.300 / **+0.073** | Sí -- mejor R², convergencia limpia |
| `receptions` | 1.354 / 0.456 | 1.477 / 0.357 | No -- peor en todo |
| `receiving_yards` | 20.42 / 0.373 | 21.39 / 0.139 | No -- peor en todo |

El enfoque jerárquico **no es una mejora universal** -- es una mejora específica donde el modelo
anterior tenía una discrepancia real de distribución (`reg:tweedie` sobre un conteo con 82% de
ceros). En los otros 2 targets, donde XGBoost ya se ajustaba razonablemente bien, la estructura
más rígida de un modelo lineal (incluso jerárquico, incluso con la familia de conteo correcta)
pierde contra la flexibilidad no paramétrica de un árbol de decisión.

**Recomendación con evidencia**: adoptar el modelo jerárquico solo para `receiving_tds`
(complementando o reemplazando al Tweedie/hurdle, decisión que sigue pendiente); mantener XGBoost
sin cambios para `receptions`/`receiving_yards`. No se guardan estos 2 modelos jerárquicos como
artefactos de producción -- quedan documentados aquí como evidencia de que se probaron, con
resultado negativo.